<a href="https://colab.research.google.com/github/AnwHus007/IPL-2025-PBI/blob/main/IPL_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile
import json
import pandas as pd
import glob

!wget -q https://cricsheet.org/downloads/ipl_json.zip -O /content/ipl_json.zip
zip_file_path = '/content/ipl_json.zip'
extraction_directory = '/content/cricsheet_data'

os.makedirs(extraction_directory, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_directory)

In [2]:
def get_ipl_2025_files(folder_path):
    print("Scanning the data lake for IPL 2025 matches")

    all_files = glob.glob(os.path.join(folder_path, "*.json"))
    ipl_2025_files = []

    for file_path in all_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
                info = data.get('info', {})

                # 1. Check the Tournament Name
                event_name = info.get('event', {}).get('name', '')

                # 2. Check the Year
                season = str(info.get('season', ''))
                match_dates = info.get('dates', [])
                is_2025 = (season == '2025') or any('2025' in date for date in match_dates)
                if event_name == "Indian Premier League" and is_2025:
                    ipl_2025_files.append(file_path)

        except Exception:
            continue

    print(f"Found {len(ipl_2025_files)} IPL 2025 matches.")

    return ipl_2025_files

In [3]:
def process_ipl_data(file_paths):
    print("Started ETL Process")

    match_list = []
    deliveries_list = []

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

            # The filename itself is the official match ID
            match_id = os.path.basename(file_path).replace('.json', '')

            # 'info' holds all the high-level match details like date, venue, and toss
            info = data.get('info', {})
            outcome = info.get('outcome', {})
            actual_winner = outcome.get('winner', outcome.get('eliminator', 'No Result'))
            match_row = {
                'Match_ID': match_id,
                'Date': info.get('dates', [None])[0],
                'City': info.get('city', 'Unknown'),
                'Venue': info.get('venue', 'Unknown'),
                'Team_1': info.get('teams', [None, None])[0],
                'Team_2': info.get('teams', [None, None])[1],
                'Toss_Winner': info.get('toss', {}).get('winner'),
                'Toss_Decision': info.get('toss', {}).get('decision'),
                'Match_Winner': actual_winner,
                'Player_of_Match': info.get('player_of_match', [None])[0]
            }
            match_list.append(match_row)

            # Now, let's dive into the ball-by-ball details
            for innings in data.get('innings', []):
                batting_team = innings.get('team')

                for over_data in innings.get('overs', []):
                    over_num = over_data.get('over')

                    for ball_idx, delivery in enumerate(over_data.get('deliveries', [])):

                        # 1. Handling Extras:
                        # We need to separate wides and no-balls from byes and leg-byes.
                        # Wides/no-balls mean the bowler has to bowl the ball again (not a legal delivery),
                        # while byes/leg-byes do not count against the bowler's runs conceded.
                        extras_dict = delivery.get('extras', {})
                        wides = 1 if 'wides' in extras_dict else 0
                        noballs = 1 if 'noballs' in extras_dict else 0
                        byes = extras_dict.get('byes', 0)
                        legbyes = extras_dict.get('legbyes', 0)

                        # 2. Handling Wickets:
                        # If the 'wickets' key exists, someone got out.
                        # We need to know 'how' they got out (Dismissal_Kind) to see if the bowler gets credit,
                        # and specifically 'who' got out (Player_Out) because the non-striker could be run out
                        is_wicket = 1 if 'wickets' in delivery else 0
                        dismissal_kind = delivery['wickets'][0].get('kind') if is_wicket else None
                        player_out = delivery['wickets'][0].get('player_out') if is_wicket else None

                        # 3. Handling Impact Players:
                        # IPL features mid-match substitutions. If a 'replacements' key pops up during a delivery,
                        # we flag it so we can analyze the impact player rule later in our dashboard.
                        has_substitution = 1 if 'replacements' in delivery else 0

                        # Construct the row for this single delivery
                        delivery_row = {
                            'Match_ID': match_id,
                            'Batting_Team': batting_team,
                            'Over_Number': over_num,
                            'Ball_Number': ball_idx + 1,
                            'Batter': delivery.get('batter'),
                            'Bowler': delivery.get('bowler'),
                            'Runs_Batter': delivery.get('runs', {}).get('batter', 0),
                            'Runs_Extras': delivery.get('runs', {}).get('extras', 0),
                            'Runs_Total': delivery.get('runs', {}).get('total', 0),
                            'Is_Wide': wides,
                            'Is_NoBall': noballs,
                            'Byes': byes,
                            'LegByes': legbyes,
                            'Is_Wicket': is_wicket,
                            'Dismissal_Kind': dismissal_kind,
                            'Player_Out': player_out,
                            'Substitution_Event': has_substitution
                        }

                        deliveries_list.append(delivery_row)

    # Convert our populated lists into Pandas DataFrames
    df_match = pd.DataFrame(match_list)
    df_deliveries = pd.DataFrame(deliveries_list)

    # Ensure the Date column is actually treated as a Date type, not just text
    df_match['Date'] = pd.to_datetime(df_match['Date'])

    print("ETL completed")
    return df_match, df_deliveries

In [4]:
from google.colab import files
import pandas as pd

file_paths = get_ipl_2025_files('/content/cricsheet_data')

if file_paths:
    df_match, df_deliveries = process_ipl_data(file_paths)

    df_match.to_csv('Match_IPL2025.csv', index=False)
    df_deliveries.to_csv('Deliveries_IPL2025.csv', index=False)
    print("CSV files saved")
else:
    print("No IPL 2025 JSON files found.")

Scanning the data lake for IPL 2025 matches
Found 74 IPL 2025 matches.
Started ETL Process
ETL completed
CSV files saved


In [5]:
df_match.head()

,Match_ID,Date,City,Venue,Team_1,Team_2,Toss_Winner,Toss_Decision,Match_Winner,Player_of_Match
0,1473500,2025-05-20,Delhi,"Arun Jaitley Stadium, Delhi",Chennai Super Kings,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,Akash Madhwal
1,1473485,2025-04-29,Delhi,"Arun Jaitley Stadium, Delhi",Kolkata Knight Riders,Delhi Capitals,Delhi Capitals,field,Kolkata Knight Riders,SP Narine
2,1473467,2025-04-14,Lucknow,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings,Chennai Super Kings,field,Chennai Super Kings,MS Dhoni
3,1473508,2025-05-29,New Chandigarh,Maharaja Yadavindra Singh International Cricke...,Punjab Kings,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,Suyash Sharma
4,1473510,2025-06-01,Ahmedabad,"Narendra Modi Stadium, Ahmedabad",Mumbai Indians,Punjab Kings,Punjab Kings,field,Punjab Kings,SS Iyer


In [6]:
df_deliveries.head()

,Match_ID,Batting_Team,Over_Number,Ball_Number,Batter,Bowler,Runs_Batter,Runs_Extras,Runs_Total,Is_Wide,Is_NoBall,Byes,LegByes,Is_Wicket,Dismissal_Kind,Player_Out,Substitution_Event
0,1473500,Chennai Super Kings,0,1,A Mhatre,TU Deshpande,1,0,1,0,0,0,0,0,None,None,0
1,1473500,Chennai Super Kings,0,2,DP Conway,TU Deshpande,2,0,2,0,0,0,0,0,None,None,0
2,1473500,Chennai Super Kings,0,3,DP Conway,TU Deshpande,4,0,4,0,0,0,0,0,None,None,0
3,1473500,Chennai Super Kings,0,4,DP Conway,TU Deshpande,0,0,0,0,0,0,0,0,None,None,0
4,1473500,Chennai Super Kings,0,5,DP Conway,TU Deshpande,0,0,0,0,0,0,0,0,None,None,0


In [7]:
import pandas as pd

def build_relational_schema(match_data, delivery_data):
    print("Transforming raw data into a clean relational model...")

    # 1. Create the Teams Table
    # We need a master list of all unique teams. We'll pull from every column that mentions a team, drop any blanks, and keep only the unique names.
    all_teams = pd.concat([
        match_data['Team_1'],
        match_data['Team_2'],
        match_data['Toss_Winner'],
        match_data['Match_Winner']
    ]).dropna().unique()

    teams_table = pd.DataFrame({'Team_Name': all_teams}).reset_index()
    teams_table['Team_ID'] = teams_table['index'] + 1
    teams_table.drop(columns=['index'], inplace=True)

    # 2. Create the Venues Table
    # Grab unique stadium and city combinations to build our location lookup.
    venues_table = match_data[['Venue', 'City']].dropna(subset=['Venue']).drop_duplicates().reset_index(drop=True).reset_index()
    venues_table['Venue_ID'] = venues_table['index'] + 1
    venues_table.drop(columns=['index'], inplace=True)

    # 3. Create the Players Table
    # This is our lookup table for batters, bowlers, players of the match, and crucially, anyone who got out (to catch non-strikers).
    all_players = pd.concat([
        delivery_data['Batter'],
        delivery_data['Bowler'],
        delivery_data['Player_Out'],
        match_data['Player_of_Match']
    ]).dropna().unique()

    players_table = pd.DataFrame({'Player_Name': all_players}).reset_index()
    players_table['Player_ID'] = players_table['index'] + 1
    players_table.drop(columns=['index'], inplace=True)

    # 4. Build Mapping Dictionaries
    team_dict = dict(zip(teams_table['Team_Name'], teams_table['Team_ID']))
    venue_dict = dict(zip(venues_table['Venue'], venues_table['Venue_ID']))
    player_dict = dict(zip(players_table['Player_Name'], players_table['Player_ID']))

    # 5. Optimize the Matches Table
    # Swap out text names to integer IDs.
    matches_relational = match_data.copy()
    matches_relational['Team_1_ID'] = matches_relational['Team_1'].map(team_dict)
    matches_relational['Team_2_ID'] = matches_relational['Team_2'].map(team_dict)
    matches_relational['Toss_Winner_ID'] = matches_relational['Toss_Winner'].map(team_dict)
    matches_relational['Match_Winner_ID'] = matches_relational['Match_Winner'].map(team_dict)
    matches_relational['Venue_ID'] = matches_relational['Venue'].map(venue_dict)
    matches_relational['Player_of_Match_ID'] = matches_relational['Player_of_Match'].map(player_dict)

    # Drop the old text columns to shrink the file size
    columns_to_drop_match = ['Team_1', 'Team_2', 'Toss_Winner', 'Match_Winner', 'Venue', 'City', 'Player_of_Match']
    matches_relational.drop(columns=columns_to_drop_match, inplace=True, errors='ignore')

    # 6. Optimize the Deliveries Table
    # Do the exact same mapping for our massive ball-by-ball dataset.
    deliveries_relational = delivery_data.copy()
    deliveries_relational['Batting_Team_ID'] = deliveries_relational['Batting_Team'].map(team_dict)
    deliveries_relational['Batter_ID'] = deliveries_relational['Batter'].map(player_dict)
    deliveries_relational['Bowler_ID'] = deliveries_relational['Bowler'].map(player_dict)
    deliveries_relational['Player_Out_ID'] = deliveries_relational['Player_Out'].map(player_dict)

    # Drop the original text columns here too
    deliveries_relational.drop(columns=['Batting_Team', 'Batter', 'Bowler', 'Player_Out'], inplace=True, errors='ignore')

    print("Data normalized successfully.")
    return teams_table, venues_table, players_table, matches_relational, deliveries_relational

# Execute the Pipeline
# Make sure to pass in the DataFrames you generated from the previous step
teams_table, venues_table, players_table, matches_relational, deliveries_relational = build_relational_schema(df_match, df_deliveries)

print("\nPlayers Table Preview:")
display(players_table.head(3))

# Export the final tables
teams_table.to_csv('Teams.csv', index=False)
venues_table.to_csv('Venues.csv', index=False)
players_table.to_csv('Players.csv', index=False)
matches_relational.to_csv('Matches.csv', index=False)
deliveries_relational.to_csv('Deliveries.csv', index=False)

Transforming raw data into a clean relational model...
Data normalized successfully.

Players Table Preview:


,Player_Name,Player_ID
0,A Mhatre,1
1,DP Conway,2
2,Urvil Patel,3
